## Model Persistence:

After finding the best model using GridSearchCV / RandomizedSearchCV:

* Training again = time waste ⏳
* Instead → save model once, reuse anytime

 This is called Model Persistence

In [3]:
import joblib

In [4]:
from Day_17.scripts.model_tuning import grid_search

joblib.dump(grid_search.best_estimator_, 'grid_search.pkl')

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
Best CV Score:   0.9631


['grid_search.pkl']

In [7]:
model = joblib.load('grid_search.pkl')

In [8]:
from Day_17.scripts.model_tuning import X

predictions = model.predict(X)

In [9]:
predictions

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0,

## Reflection:


### How many models will GridSearchCV train?

If you have 5 hyperparameters, each with 10 options:

10^5=100,000 combinations

With k-fold cross-validation (e.g., cv = 3):

100,000×3=300,000 model fits

So, GridSearchCV would train 300,000 models


### Why use RandomizedSearchCV for huge datasets?

GridSearch grows exponentially with more parameters
Training lakhs of models becomes:
* Very slow
* Computationally expensive

RandomizedSearchCV solves this by:

* Sampling only a fixed number of combinations (n_iter)
* Reducing training from lakhs → just a few hundred or less
* Still achieving near-optimal performance

In [1]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
import numpy as np

# Load data
data = load_breast_cancer()
X, y = data.data, data.target

# Model
rf = RandomForestClassifier(random_state=42)

# Parameter distribution (not grid)
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10, 15]
}

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=10,              # 🔥 only 10 random trials
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Train
random_search.fit(X, y)

# Output
print("Best Parameters:", random_search.best_params_)
print("Best Score:", random_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameters: {'n_estimators': 50, 'min_samples_split': 2, 'max_depth': None}
Best Score: 0.9631021999443052


## **Observation:**

* Achieved same accuracy (96.31%) as GridSearchCV.
* Used fewer model trainings (30 vs 81 fits).
* Reduced computation by ~63%.
* Avoided testing unnecessary parameter combinations.
* Provided faster execution time.
* More scalable for larger datasets and parameter spaces.
* Controlled search using n_iter (fixed trials).
* Efficiently explored good regions of hyperparameter space.

In [5]:
# Save the best model
joblib.dump(random_search.best_estimator_, 'randomize_search_breast_Cancer_model.pkl')

print(" Best model saved as randomize_search_best_model.pkl")

 Best model saved as randomize_search_best_model.pkl
